# Comparing Two TEI Editions of Aristotle's *Poetics*

This notebook compares:
- **Butcher 1911 Greek** (`tlg0086_tlg034_butcher1911-grc.xml`) — LLM-produced TEI with `<lb>` line breaks and end-of-line hyphens
- **Perseus/Kassel Greek** (`tlg0086_tlg034_perseus-grc2.xml`) — the reference Perseus Digital Library edition
- **Butcher 1911 English** (`tlg0086_tlg034_butcher1911-eng2.xml`) — Butcher's English translation, aligned by chapter/section

**Part A** — Greek vs. Greek: verify no words were dropped and chapter/section divs match
**Part B** — Greek + English: side-by-side display for aligned sections

In [68]:
import re, unicodedata, json
from collections import OrderedDict, defaultdict
from lxml import etree
from difflib import SequenceMatcher
from IPython.display import display, HTML, Markdown

## 0. File paths

Edit these paths to match your local setup.

In [69]:
# ── EDIT THESE PATHS ──
BUTCHER_GRC = '/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-grc.xml'
PERSEUS_GRC = '/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml'
BUTCHER_ENG = '/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml'

## 1. Parse all XML files

In [70]:
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

butcher_tree = etree.parse(BUTCHER_GRC)
perseus_tree = etree.parse(PERSEUS_GRC)
english_tree = etree.parse(BUTCHER_ENG)

print('Butcher Greek root tag:', butcher_tree.getroot().tag)
print('Perseus Greek root tag:', perseus_tree.getroot().tag)
print('Butcher English root tag:', english_tree.getroot().tag)

Butcher Greek root tag: {http://www.tei-c.org/ns/1.0}TEI
Perseus Greek root tag: {http://www.tei-c.org/ns/1.0}TEI
Butcher English root tag: {http://www.tei-c.org/ns/1.0}TEI


## 2. Text extraction and normalization

The Butcher Greek has `<lb>` line breaks and **end-of-line hyphens** (e.g. `πρῶ-` / `τον`).
Both files have `<del>`, `<add>`, `<quote>`, `[cite: N]` markers, and `<note>` elements.

In [71]:
def extract_text_from_element(el):
    """Recursively extract all text content from an XML element,
    including text inside <add>, <del>, <quote> children,
    but EXCLUDING <note> elements.
    Treats <lb/>, <pb/>, <milestone/> as space boundaries.
    """
    parts = []

    def _walk(node):
        tag = etree.QName(node.tag).localname if isinstance(node.tag, str) else ''
        if tag == 'note':
            if node.tail:
                parts.append(node.tail)
            return
        if tag in ('lb', 'pb', 'milestone'):
            parts.append(' ')
            if node.tail:
                parts.append(node.tail)
            return
        if node.text:
            parts.append(node.text)
        for child in node:
            _walk(child)
        if node.tail:
            parts.append(node.tail)

    _walk(el)
    return ''.join(parts)


def normalize_greek(text):
    """Normalize extracted Greek text for comparison."""
    text = re.sub(r'\[cite:\s*\d+\]', '', text)
    text = re.sub(r'(\S)-\s+', r'\1', text)
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u2019', '\'')
    text = text.replace('\u02BC', '\'')
    text = text.replace('\u1FBD', '\'')
    text = text.replace('\u2018', '')
    text = re.sub(r'\.\s*\.\s*\.\s*\.?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_english(text):
    """Normalize extracted English text."""
    text = re.sub(r'\[cite:\s*\d+\]', '', text)
    text = re.sub(r'(\S)-\s+', r'\1', text)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def section_sort_key(s):
    """Sort key for section numbers like '11b', '10c'."""
    m = re.match(r'(\d+)(.*)', s)
    if m:
        return (int(m.group(1)), m.group(2))
    return (0, s)


def tokenize(text):
    """Split text into word tokens, stripping boundary punctuation."""
    tokens = text.split()
    cleaned = []
    for t in tokens:
        t = re.sub(r'^[^\w\u0370-\u03FF\u1F00-\u1FFF]+', '', t)
        t = re.sub(r'[^\w\u0370-\u03FF\u1F00-\u1FFF]+$', '', t)
        if t:
            cleaned.append(t)
    return cleaned


def chapter_full_text(ch_dict):
    """Concatenate all sections of a chapter in section order."""
    return ' '.join(ch_dict[s] for s in sorted(ch_dict.keys(), key=section_sort_key))


# Quick test
test = "πρῶ- [cite: 4]\n            τον ἀπὸ τῶν πρώτων. [cite: 4]"
print(repr(normalize_greek(test)))

'πρῶτον ἀπὸ τῶν πρώτων.'


## 3. Build chapter → section dictionaries

In [72]:
def build_chapter_dict(tree, ch_tag_attr, sec_tag_attr):
    """Generic builder for both Butcher and Perseus XML structures.
    ch_tag_attr / sec_tag_attr are (attribute_name, attribute_value) tuples.
    Duplicate chapter divs (Butcher page-break splits) are merged.
    """
    ch_attr, ch_val = ch_tag_attr
    sec_attr, sec_val = sec_tag_attr
    chapters = OrderedDict()
    for ch_div in tree.xpath(f'//tei:div[@{ch_attr}="{ch_val}"]', namespaces=NS):
        ch_n = ch_div.get('n')
        if ch_n not in chapters:
            chapters[ch_n] = OrderedDict()
        for sec_div in ch_div.xpath(f'tei:div[@{sec_attr}="{sec_val}"]', namespaces=NS):
            sec_n = sec_div.get('n')
            raw = extract_text_from_element(sec_div)
            chapters[ch_n][sec_n] = raw
    return chapters


def build_and_normalize(tree, ch_tag, sec_tag, normalizer):
    raw = build_chapter_dict(tree, ch_tag, sec_tag)
    return OrderedDict(
        (ch, OrderedDict((s, normalizer(txt)) for s, txt in secs.items()))
        for ch, secs in raw.items()
    )


butcher = build_and_normalize(butcher_tree, ('type', 'chapter'), ('type', 'section'), normalize_greek)
perseus = build_and_normalize(perseus_tree, ('subtype', 'chapter'), ('subtype', 'subchapter'), normalize_greek)
english = build_and_normalize(english_tree, ('type', 'chapter'), ('type', 'section'), normalize_english)

print(f'Butcher Greek:   {len(butcher)} chapters, {sum(len(s) for s in butcher.values())} sections')
print(f'Perseus Greek:   {len(perseus)} chapters, {sum(len(s) for s in perseus.values())} sections')
print(f'Butcher English: {len(english)} chapters, {sum(len(s) for s in english.values())} sections')

Butcher Greek:   26 chapters, 381 sections
Perseus Greek:   26 chapters, 381 sections
Butcher English: 26 chapters, 381 sections


---
# Part A — Greek vs. Greek Comparison

## 4. Compare chapter/section structure

In [73]:
all_chapters = sorted(set(butcher.keys()) | set(perseus.keys()), key=section_sort_key)

rows = []
for ch in all_chapters:
    b_secs = sorted(butcher.get(ch, {}).keys(), key=section_sort_key)
    p_secs = sorted(perseus.get(ch, {}).keys(), key=section_sort_key)
    b_set, p_set = set(b_secs), set(p_secs)
    only_b = sorted(b_set - p_set, key=section_sort_key)
    only_p = sorted(p_set - b_set, key=section_sort_key)
    match = '✅' if b_set == p_set else '❌'
    rows.append({
        'chapter': ch, 'butcher_sections': len(b_secs), 'perseus_sections': len(p_secs),
        'match': match, 'only_in_butcher': ', '.join(only_b), 'only_in_perseus': ', '.join(only_p),
    })

html = '<table border="1" cellpadding="4" style="border-collapse:collapse">'
html += '<tr><th>Chapter</th><th>Butcher §</th><th>Perseus §</th><th>Match</th><th>Only Butcher</th><th>Only Perseus</th></tr>'
for r in rows:
    color = '' if r['match'] == '✅' else ' style="background:#ffe0e0"'
    html += f'<tr{color}><td>{r["chapter"]}</td><td>{r["butcher_sections"]}</td><td>{r["perseus_sections"]}</td>'
    html += f'<td>{r["match"]}</td><td>{r["only_in_butcher"]}</td><td>{r["only_in_perseus"]}</td></tr>'
html += '</table>'
display(HTML(html))

Chapter,Butcher §,Perseus §,Match,Only Butcher,Only Perseus
1,14,14,✅,,
2,7,7,✅,,
3,7,7,✅,,
4,21,21,✅,,
5,11,11,✅,,
6,28,28,✅,,
7,12,12,✅,,
8,4,4,✅,,
9,15,15,✅,,
10,4,4,✅,,


## 5. Word-level comparison per chapter

In [74]:
def compare_word_lists(words_a, words_b):
    """Returns (only_in_a, only_in_b, common_count, ratio)."""
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    only_a, only_b, common = [], [], 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            common += (i2 - i1)
        elif tag == 'delete':
            only_a.extend((k, words_a[k]) for k in range(i1, i2))
        elif tag == 'insert':
            only_b.extend((k, words_b[k]) for k in range(j1, j2))
        elif tag == 'replace':
            only_a.extend((k, words_a[k]) for k in range(i1, i2))
            only_b.extend((k, words_b[k]) for k in range(j1, j2))
    return only_a, only_b, common, sm.ratio()

In [75]:
chapter_results = []
for ch in all_chapters:
    b_text = chapter_full_text(butcher[ch]) if ch in butcher else ''
    p_text = chapter_full_text(perseus[ch]) if ch in perseus else ''
    b_words, p_words = tokenize(b_text), tokenize(p_text)
    only_b, only_p, common, ratio = compare_word_lists(b_words, p_words)
    chapter_results.append({
        'chapter': ch, 'butcher_words': len(b_words), 'perseus_words': len(p_words),
        'common': common, 'only_butcher': only_b, 'only_perseus': only_p, 'ratio': ratio,
    })

html = '<h3>Word-level comparison per chapter</h3>'
html += '<table border="1" cellpadding="4" style="border-collapse:collapse; font-size:13px">'
html += '<tr><th>Ch</th><th>Butcher</th><th>Perseus</th><th>Shared</th><th>Only B</th><th>Only P</th><th>Sim</th></tr>'
for r in chapter_results:
    ob, op = len(r['only_butcher']), len(r['only_perseus'])
    color = ''
    if ob > 0 or op > 0:
        color = ' style="background:#fff3cd"' if r['ratio'] > 0.95 else ' style="background:#ffe0e0"'
    html += f'<tr{color}><td>{r["chapter"]}</td><td>{r["butcher_words"]}</td><td>{r["perseus_words"]}</td>'
    html += f'<td>{r["common"]}</td><td>{ob}</td><td>{op}</td><td>{r["ratio"]:.3f}</td></tr>'
html += '</table>'
display(HTML(html))

Ch,Butcher,Perseus,Shared,Only B,Only P,Sim
1,407,406,399,8,7,0.982
2,161,161,159,2,2,0.988
3,203,203,201,2,2,0.990
4,570,569,548,22,21,0.962
5,237,236,233,4,3,0.985
6,713,715,689,24,26,0.965
7,318,319,314,4,5,0.986
8,188,188,186,2,2,0.989
9,476,477,468,8,9,0.982
10,90,90,86,4,4,0.956


## 6. Detailed chapter diffs

In [76]:
for r in chapter_results:
    if not r['only_butcher'] and not r['only_perseus']:
        continue
    ch = r['chapter']
    print(f"\n{'='*70}")
    print(f"Chapter {ch}  (similarity: {r['ratio']:.3f})")
    print(f"{'='*70}")
    if r['only_butcher']:
        print(f"\n  Words ONLY in Butcher ({len(r['only_butcher'])}):\n")
        for pos, w in r['only_butcher']:
            print(f"    [{pos:4d}] {w}")
    if r['only_perseus']:
        print(f"\n  Words ONLY in Perseus ({len(r['only_perseus'])}):\n")
        for pos, w in r['only_perseus']:
            print(f"    [{pos:4d}] {w}")


Chapter 1  (similarity: 0.982)

  Words ONLY in Butcher (8):

    [   0] Περὶ
    [ 201] ἢ
    [ 208] μετ᾿
    [ 217] ἀνώνυμος
    [ 218] τυγχάνει
    [ 219] οὖσα
    [ 263] ἐλεγειοποιούς
    [ 339] τοῦτον

  Words ONLY in Perseus (7):

    [   0] περὶ
    [ 201] καὶ
    [ 202] ἡ
    [ 209] μετ
    [ 218] ἀνώνυμοι
    [ 219] τυγχάνουσι
    [ 263] ἐλεγειοποιοὺς

Chapter 2  (similarity: 0.988)

  Words ONLY in Butcher (2):

    [   0] Ἐπεὶ
    [ 139] τῇ

  Words ONLY in Perseus (2):

    [   0] ἐπεὶ
    [ 141] τῇ

Chapter 3  (similarity: 0.990)

  Words ONLY in Butcher (2):

    [   0] Ἔτι
    [  57] κατ᾿

  Words ONLY in Perseus (2):

    [   0] ἔτι
    [  57] κατ

Chapter 4  (similarity: 0.962)

  Words ONLY in Butcher (22):

    [   0] Ἐοίκασι
    [  22] ἐστί
    [ 141] δὴ
    [ 149] και
    [ 160] φανερόν
    [ 164] καὶ
    [ 206] ἅτεροι
    [ 240] ἁρμόττον
    [ 243] μέτρον
    [ 244] διὸ
    [ 282] ὅτι
    [ 289] τὰ
    [ 292] σχήματα
    [ 351] μείζονα
    [ 364] ἄρ
    [ 378] κρ

## 7. Section-level comparison — full word accounting

Every section in either edition is shown. Sections only in one edition appear in full
so no words are silently skipped. Totals at the end confirm every word is accounted for.

In [77]:
def html_word_diff(text_a, text_b, label_a='Butcher', label_b='Perseus'):
    words_a, words_b = text_a.split(), text_b.split()
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    html_a, html_b = [], []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            chunk = ' '.join(words_a[i1:i2])
            html_a.append(chunk); html_b.append(chunk)
        elif tag == 'delete':
            html_a.append(f'<span style="background:#ffa0a0;font-weight:bold">{" ".join(words_a[i1:i2])}</span>')
        elif tag == 'insert':
            html_b.append(f'<span style="background:#a0ffa0;font-weight:bold">{" ".join(words_b[j1:j2])}</span>')
        elif tag == 'replace':
            html_a.append(f'<span style="background:#ffa0a0;font-weight:bold">{" ".join(words_a[i1:i2])}</span>')
            html_b.append(f'<span style="background:#a0ffa0;font-weight:bold">{" ".join(words_b[j1:j2])}</span>')
    return (f'<div style="margin:4px 0"><b>{label_a}:</b> {" ".join(html_a)}</div>'
            f'<div style="margin:4px 0"><b>{label_b}:</b> {" ".join(html_b)}</div>')

In [78]:
diff_count = 0
identical_count = 0
only_butcher_secs = 0
only_perseus_secs = 0
words_compared_b = 0
words_compared_p = 0

html_parts = ['<h3>Section-level comparison (all sections, both editions)</h3>']

for ch in all_chapters:
    b_secs = set(butcher.get(ch, {}).keys())
    p_secs = set(perseus.get(ch, {}).keys())
    all_secs = sorted(b_secs | p_secs, key=section_sort_key)

    for sec in all_secs:
        in_b = sec in b_secs
        in_p = sec in p_secs
        if in_b:
            words_compared_b += len(butcher[ch][sec].split())
        if in_p:
            words_compared_p += len(perseus[ch][sec].split())

        if in_b and in_p:
            b_text, p_text = butcher[ch][sec], perseus[ch][sec]
            if b_text == p_text:
                identical_count += 1
                continue
            diff_count += 1
            html_parts.append(
                f'<div style="border:1px solid #ccc; padding:8px; margin:8px 0; border-radius:4px">'
                f'<b>Ch {ch} §{sec}</b><br>{html_word_diff(b_text, p_text)}</div>')
        elif in_b:
            only_butcher_secs += 1
            html_parts.append(
                f'<div style="border:1px solid #e88; padding:8px; margin:8px 0; border-radius:4px; background:#fff5f5">'
                f'<b>Ch {ch} §{sec}</b> — <em>only in Butcher</em><br>'
                f'<div style="margin:4px 0">{butcher[ch][sec]}</div></div>')
        else:
            only_perseus_secs += 1
            html_parts.append(
                f'<div style="border:1px solid #8e8; padding:8px; margin:8px 0; border-radius:4px; background:#f5fff5">'
                f'<b>Ch {ch} §{sec}</b> — <em>only in Perseus</em><br>'
                f'<div style="margin:4px 0">{perseus[ch][sec]}</div></div>')

# Word accounting
butcher_total = sum(len(t.split()) for ch in butcher.values() for t in ch.values())
perseus_total = sum(len(t.split()) for ch in perseus.values() for t in ch.values())

b_check = '✅' if words_compared_b == butcher_total else '❌ MISMATCH'
p_check = '✅' if words_compared_p == perseus_total else '❌ MISMATCH'

summary = (
    f'<p>Sections: <b>{diff_count + identical_count}</b> shared '
    f'({identical_count} identical, {diff_count} with diffs), '
    f'<b>{only_butcher_secs}</b> only-Butcher, '
    f'<b>{only_perseus_secs}</b> only-Perseus</p>'
    f'<p>Word accounting — Butcher: {words_compared_b}/{butcher_total} ({b_check}), '
    f'Perseus: {words_compared_p}/{perseus_total} ({p_check})</p>')
html_parts.insert(1, summary)
display(HTML('\n'.join(html_parts)))

## 8. Overall statistics

In [79]:
butcher_full = ' '.join(chapter_full_text(butcher[ch]) for ch in sorted(butcher.keys(), key=section_sort_key))
perseus_full = ' '.join(chapter_full_text(perseus[ch]) for ch in sorted(perseus.keys(), key=section_sort_key))
b_all, p_all = tokenize(butcher_full), tokenize(perseus_full)
only_b, only_p, common, ratio = compare_word_lists(b_all, p_all)

print(f'Total Butcher words:  {len(b_all)}')
print(f'Total Perseus words:  {len(p_all)}')
print(f'Shared (aligned):     {common}')
print(f'Only in Butcher:      {len(only_b)}')
print(f'Only in Perseus:      {len(only_p)}')
print(f'Overall similarity:   {ratio:.4f}')

Total Butcher words:  10252
Total Perseus words:  10256
Shared (aligned):     9935
Only in Butcher:      317
Only in Perseus:      321
Overall similarity:   0.9689


## 9. Classify differences

In [80]:
def strip_accents(s):
    nfd = unicodedata.normalize('NFD', s)
    return ''.join(c for c in nfd if unicodedata.category(c) not in ('Mn',))


def classify_opcodes(words_a, words_b):
    sm = SequenceMatcher(None, words_a, words_b, autojunk=False)
    results = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            continue
        a_chunk, b_chunk = words_a[i1:i2], words_b[j1:j2]
        if tag == 'replace' and len(a_chunk) == len(b_chunk):
            for wa, wb in zip(a_chunk, b_chunk):
                kind = 'orthographic' if strip_accents(wa) == strip_accents(wb) else 'variant'
                results.append({'type': kind, 'butcher': wa, 'perseus': wb})
        elif tag == 'replace':
            results.append({'type': 'variant', 'butcher': ' '.join(a_chunk), 'perseus': ' '.join(b_chunk)})
        elif tag == 'delete':
            results.append({'type': 'only_butcher', 'butcher': ' '.join(a_chunk), 'perseus': ''})
        elif tag == 'insert':
            results.append({'type': 'only_perseus', 'butcher': '', 'perseus': ' '.join(b_chunk)})
    return results


all_diffs = classify_opcodes(b_all, p_all)
counts = defaultdict(int)
for d in all_diffs:
    counts[d['type']] += 1

print('Difference classification:')
for k in ['orthographic', 'variant', 'only_butcher', 'only_perseus']:
    print(f'  {k:20s}: {counts[k]:4d}')

Difference classification:
  orthographic        :   60
  variant             :  137
  only_butcher        :   36
  only_perseus        :   44


In [81]:
html = '<h3>All differences (classified)</h3>'
html += '<div style="max-height:600px; overflow-y:auto">'
html += '<table border="1" cellpadding="4" style="border-collapse:collapse; font-size:13px">'
html += '<tr><th>#</th><th>Type</th><th>Butcher</th><th>Perseus</th></tr>'
type_colors = {'orthographic': '#e8f4fd', 'variant': '#fff3cd', 'only_butcher': '#ffd6d6', 'only_perseus': '#d6ffd6'}
for i, d in enumerate(all_diffs, 1):
    bg = type_colors.get(d['type'], '#fff')
    html += f'<tr style="background:{bg}"><td>{i}</td><td>{d["type"]}</td>'
    html += f'<td>{d["butcher"]}</td><td>{d["perseus"]}</td></tr>'
html += '</table></div>'
display(HTML(html))

#,Type,Butcher,Perseus
1,variant,Περὶ,περὶ
2,variant,ἢ,καὶ ἡ
3,variant,μετ᾿,μετ
4,variant,ἀνώνυμος τυγχάνει οὖσα,ἀνώνυμοι τυγχάνουσι
5,orthographic,ἐλεγειοποιούς,ἐλεγειοποιοὺς
6,only_butcher,τοῦτον,
7,variant,Ἐπεὶ,ἐπεὶ
8,only_butcher,τῇ,
9,only_perseus,,τῇ
10,variant,Ἔτι,ἔτι


## 10. Check for hyphen-rejoin failures

In [82]:
stray_hyphens = []
for ch in sorted(butcher.keys(), key=section_sort_key):
    for sec in sorted(butcher[ch].keys(), key=section_sort_key):
        for m in re.finditer(r'\S*-\S*', butcher[ch][sec]):
            w = m.group()
            if w not in ('—', '–') and not w.startswith('—'):
                stray_hyphens.append((ch, sec, w))

if stray_hyphens:
    print(f'⚠️  {len(stray_hyphens)} remaining hyphens:')
    for ch, sec, w in stray_hyphens[:30]:
        print(f'  Ch {ch}, §{sec}: "{w}"')
else:
    print('✅ No stray hyphens — all line-break hyphens successfully rejoined.')

✅ No stray hyphens — all line-break hyphens successfully rejoined.


## 11. Part A Summary

In [83]:
print('=' * 60)
print('PART A — GREEK vs. GREEK SUMMARY')
print('=' * 60)
b_chs, p_chs = set(butcher.keys()), set(perseus.keys())
if b_chs == p_chs:
    print(f'\n✅ Chapters: Both have {len(b_chs)} chapters')
else:
    print(f'\n❌ Chapter mismatch!')
    if b_chs - p_chs: print(f'   Only in Butcher: {b_chs - p_chs}')
    if p_chs - b_chs: print(f'   Only in Perseus: {p_chs - b_chs}')
mismatched = [r for r in rows if r['match'] == '❌']
if mismatched:
    print(f'\n⚠️  {len(mismatched)} chapters have different section counts')
    for r in mismatched:
        print(f'   Ch {r["chapter"]}: B={r["butcher_sections"]}, P={r["perseus_sections"]}')
print(f'\n📊 Words: B={len(b_all)}, P={len(p_all)}, similarity={ratio:.4f}')
print(f'   Orthographic: {counts["orthographic"]}, Variants: {counts["variant"]}')
print(f'   Only-B groups: {counts["only_butcher"]}, Only-P groups: {counts["only_perseus"]}')
if stray_hyphens:
    print(f'\n⚠️  Stray hyphens: {len(stray_hyphens)}')
else:
    print(f'\n✅ All hyphens rejoined')

PART A — GREEK vs. GREEK SUMMARY

✅ Chapters: Both have 26 chapters

📊 Words: B=10252, P=10256, similarity=0.9689
   Orthographic: 60, Variants: 137
   Only-B groups: 36, Only-P groups: 44

✅ All hyphens rejoined


---
# Part B — Greek + English Alignment

Side-by-side display of Butcher's Greek text and English translation, section by section.

## 12. Greek ↔ English structure comparison

In [84]:
grc_chs = set(butcher.keys())
eng_chs = set(english.keys())

print(f'Greek chapters:   {sorted(grc_chs, key=section_sort_key)}')
print(f'English chapters: {sorted(eng_chs, key=section_sort_key)}')
print()
if grc_chs == eng_chs:
    print('✅ Same chapters')
else:
    if grc_chs - eng_chs:
        print(f'⚠️  In Greek but NOT English: {sorted(grc_chs - eng_chs, key=section_sort_key)}')
    if eng_chs - grc_chs:
        print(f'⚠️  In English but NOT Greek: {sorted(eng_chs - grc_chs, key=section_sort_key)}')

ge_rows = []
for ch in sorted(grc_chs | eng_chs, key=section_sort_key):
    g_secs = set(butcher.get(ch, {}).keys())
    e_secs = set(english.get(ch, {}).keys())
    only_g = sorted(g_secs - e_secs, key=section_sort_key)
    only_e = sorted(e_secs - g_secs, key=section_sort_key)
    match = '✅' if g_secs == e_secs else '❌'
    ge_rows.append({
        'chapter': ch, 'grc_secs': len(g_secs), 'eng_secs': len(e_secs),
        'match': match, 'only_grc': ', '.join(only_g), 'only_eng': ', '.join(only_e),
    })

html = '<table border="1" cellpadding="4" style="border-collapse:collapse">'
html += '<tr><th>Ch</th><th>Greek §</th><th>English §</th><th>Match</th><th>Only Greek</th><th>Only English</th></tr>'
for r in ge_rows:
    color = '' if r['match'] == '✅' else ' style="background:#ffe0e0"'
    html += f'<tr{color}><td>{r["chapter"]}</td><td>{r["grc_secs"]}</td><td>{r["eng_secs"]}</td>'
    html += f'<td>{r["match"]}</td><td>{r["only_grc"]}</td><td>{r["only_eng"]}</td></tr>'
html += '</table>'
display(HTML(html))

Greek chapters:   ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26']
English chapters: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26']

✅ Same chapters


Ch,Greek §,English §,Match,Only Greek,Only English
1,14,14,✅,,
2,7,7,✅,,
3,7,7,✅,,
4,21,21,✅,,
5,11,11,✅,,
6,28,28,✅,,
7,12,12,✅,,
8,4,4,✅,,
9,15,15,✅,,
10,4,4,✅,,


## 13. Side-by-side Greek + English (all chapters)

Greek on the left, English on the right. Missing sections are highlighted.

In [85]:
html_parts = []

for ch in sorted(grc_chs | eng_chs, key=section_sort_key):
    g_dict = butcher.get(ch, {})
    e_dict = english.get(ch, {})
    all_secs = sorted(set(g_dict.keys()) | set(e_dict.keys()), key=section_sort_key)

    html_parts.append(
        f'<h3 style="margin-top:24px; border-bottom:2px solid #333; padding-bottom:4px">'
        f'Chapter {ch}</h3>'
        '<table border="1" cellpadding="8" style="border-collapse:collapse; width:100%; table-layout:fixed">'
        '<tr style="background:#f0f0f0"><th style="width:8%">§</th>'
        '<th style="width:46%">Greek (Butcher)</th>'
        '<th style="width:46%">English (Butcher)</th></tr>')

    for sec in all_secs:
        g_text = g_dict.get(sec, '')
        e_text = e_dict.get(sec, '')
        if g_text and not e_text:
            g_style = 'background:#fff5f5'
            e_style = 'background:#fff5f5; color:#aaa; font-style:italic'
            e_text = '(no English for this section)'
        elif e_text and not g_text:
            g_style = 'background:#f5fff5; color:#aaa; font-style:italic'
            e_style = 'background:#f5fff5'
            g_text = '(no Greek for this section)'
        else:
            g_style = ''
            e_style = ''
        html_parts.append(
            f'<tr><td style="vertical-align:top; text-align:center; font-weight:bold">{sec}</td>'
            f'<td style="vertical-align:top; {g_style}">{g_text}</td>'
            f'<td style="vertical-align:top; {e_style}">{e_text}</td></tr>')

    html_parts.append('</table>')

display(HTML('\n'.join(html_parts)))

§,Greek (Butcher),English (Butcher)
1,"Περὶ ποιητικῆς αὐτῆς τε καὶ τῶν εἰδῶν αὐτῆς ἥν τινα δύναμιν ἕκαστον ἔχει, καὶ πῶς δεῖ συνίστασθαι τοὺς μύθους εἰ μέλλει καλῶς ἕξειν ἡ ποίησις, ἔτι δὲ ἐκ πόσων καὶ ποίων ἐστὶ μορίων, ὁμοίως δὲ καὶ περὶ τῶν ἄλλων ὅσα τῆς αὐτῆς ἐστι μεθόδου, λέγωμεν ἀρξάμενοι κατὰ φύσιν πρῶτον ἀπὸ τῶν πρώτων.","I propose to treat of Poetry in itself and of its various kinds, noting the essential quality of each; to inquire into the structure of the plot as requisite to a good poem; into the number and nature of the parts of which a poem is composed; and similarly into whatever else falls within the same inquiry. Following, then, the order of nature, let us begin with the principles which come first."
2,"ἐποποιία δὴ καὶ ἡ τῆς τραγῳδίας ποίησις ἔτι δὲ κωμῳδία καὶ ἡ διθυραμβοποιητικὴ καὶ τῆς αὐλητικῆς ἡ πλείστη καὶ κιθαριστικῆς πᾶσαι τυγχάνουσιν οὖσαι μιμήσεις τὸ σύνολον,","Epic poetry and Tragedy, Comedy also and Dithyrambic poetry, and the music of the flute and of the lyre in most of their forms, are all in their general conception modes of imitation."
3,"διαφέρουσι δὲ ἀλλήλων τρισίν, ἢ γὰρ τῷ ἐν ἑτέροις μιμεῖσθαι ἢ τῷ ἕτερα ἢ τῷ ἑτέρως καὶ μὴ τὸν αὐτὸν τρόπον.","They differ, however, from one another in three respects,—the medium, the objects, the manner or mode of imitation, being in each case distinct."
4,"ὥσπερ γὰρ καὶ χρώμασι καὶ σχήμασι πολλὰ μιμοῦνταί τινες ἀπεικάζοντες (οἱ μὲν διὰ τέχνης οἱ δὲ διὰ συνηθείας), ἕτεροι δὲ διὰ τῆς φωνῆς, οὕτω κἀν ταῖς εἰρημέναις τέχναις· ἅπασαι μὲν ποιοῦνται τὴν μίμησιν ἐν ῥυθμῷ καὶ λόγῳ καὶ ἁρμονίᾳ, τούτοις δ' ἢ χωρὶς ἢ μεμιγμένοις·","For as there are persons who, by conscious art or mere habit, imitate and represent various objects through the medium of colour and form, or again by the voice; so in the arts above mentioned, taken as a whole, the imitation is produced by rhythm, language, or 'harmony,' either singly or combined."
5,"οἷον ἁρμονίᾳ μὲν καὶ ῥυθμῷ χρώμεναι μόνον ἥ τε αὐλητικὴ καὶ ἡ κιθαριστικὴ κἂν εἴ τινες ἕτεραι τυγχάνωσιν οὖσαι τοιαῦται τὴν δύναμιν, οἷον ἡ τῶν συρίγγων·","Thus in the music of the flute and of the lyre, 'harmony' and rhythm alone are employed; also in other arts, such as that of the shepherd's pipe, which are essentially similar to these."
6,"αὐτῷ δὲ τῷ ῥυθμῷ [μιμοῦνται] χωρὶς ἁρμονίας ἡ τῶν ὀρχηστῶν, καὶ γὰρ οὗτοι διὰ τῶν σχηματιζομένων ῥυθμῶν μιμοῦνται καὶ ἤθη καὶ πάθη καὶ πράξεις·","In dancing, rhythm alone is used without 'harmony'; for even dancing imitates character, emotion, and action, by rhythmical movement."
7,"ἡ δὲ [ἐποποιία] μόνον τοῖς λόγοις ψιλοῖς ἢ τοῖς μέτροις καὶ τούτοις εἴτε μιγνῦσα μετ᾿ ἀλλήλων εἴθ' ἑνί τινι γένει χρωμένη τῶν μέτρων, <ἀνώνυμος> τυγχάνει οὖσα μέχρι τοῦ νῦν·","There is another art which imitates by means of language alone, and that either in prose or verse—which verse, again, may either combine different metres or consist of but one kind—but this has hitherto been without a name."
8,"οὐδὲν γὰρ ἂν ἔχοιμεν ὀνομάσαι κοινὸν τοὺς Σώφρονος καὶ Ξενάρχου μίμους καὶ τοὺς Σωκρατικοὺς λόγους,",For there is no common term we could apply to the mimes of Sophron and Xenarchus and the Socratic dialogues on the one hand;
9,οὐδὲ εἴ τις διὰ τριμέτρων ἢ ἐλεγείων ἢ τῶν ἄλλων τινῶν τῶν τοιούτων ποιοῖτο τὴν μίμησιν·,"and, on the other, to poetic imitations in iambic, elegiac, or any similar metre."
10,"πλὴν οἱ ἄνθρωποί γε συνάπτοντες τῷ μέτρῳ τὸ ποιεῖν ἐλεγειοποιούς, τοὺς δὲ ἐποποιοὺς ὀνομάζουσιν, οὐχ ὡς κατὰ τὴν μίμησιν ποιητὰς ἀλλὰ κοινῇ κατὰ τὸ μέτρον προσαγορεύοντες.","People do, indeed, add the word 'maker' or 'poet' to the name of the metre, and speak of elegiac poets, or epic (that is, hexameter) poets, as if it were not the imitation that makes the poet, but the verse that entitles them all indiscriminately to the name."


## 13a. Word-level alignment — setup

We generate word-level alignments between the Butcher Greek and English using the
Anthropic API, following the LLM-based approach from
[Nadel & Crane (2025)](https://doi.org/10.1017/chr.2025.10010).

Their evaluation showed Claude 3.5 Sonnet achieved F1 ≈ 0.85 on gold-standard
Ancient Greek → English alignment (Ugarit data), outperforming the UGARIT/grc-alignment
XLM-RoBERTa model (Yousef et al. 2022). We use a similar prompt design with two
refinements for **high precision**:

1. **Confidence tagging** — the model marks each alignment as `sure` or `possible`,
   and we keep only `sure` alignments for the final display.
2. **JSON output** — structured output avoids the formatting fragility noted in the paper.

You need an `ANTHROPIC_API_KEY` environment variable (or set it below).


In [ ]:
import anthropic, time, json as json_mod, os
from IPython.display import display, HTML

# ── Configuration ──
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"
ALIGNMENT_CACHE_FILE = "poetics_alignments.json"
ONLY_SURE = True          # True = high-precision (sure only); False = include possible
RATE_LIMIT_DELAY = 1.0    # seconds between API calls

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env

# ── Prompt template ──
# Adapted from Nadel & Crane (2025), switched to JSON output for reliability.

SYSTEM_PROMPT = """You are an expert in Ancient Greek (polytonic) and English. Your task is to produce word-level translation alignments between an Ancient Greek sentence and its English translation.

Rules:
- Every Greek word should be aligned to one or more English words, or to null if it has no direct English counterpart (e.g. particles, articles with no English match).
- Every English word should appear in exactly one alignment entry.
- Greek function words (articles, particles, conjunctions) that map to English words should still be aligned; only use null when there truly is no English counterpart.
- Mark each alignment as "sure" (confident, clear semantic correspondence) or "possible" (ambiguous, could reasonably be aligned differently).

Respond with ONLY a JSON array. Each element is an object:
  {"greek": "<Greek word(s)>", "english": "<English word(s) or null>", "confidence": "sure"|"possible"}

Do not include any commentary, markdown, or text outside the JSON array."""

FEW_SHOT_EXAMPLES = [
    {
        "greek": "\u039c\u1fc6\u03bd\u03b9\u03bd \u1f04\u03b5\u03b9\u03b4\u03b5 \u03b8\u03b5\u1f70 \u03a0\u03b7\u03bb\u03b7\u03ca\u03ac\u03b4\u03b5\u03c9 \u1f08\u03c7\u03b9\u03bb\u1fc6\u03bf\u03c2",
        "english": "Sing, O goddess, the anger of Achilles son of Peleus",
        "alignment": [
            {"greek": "\u039c\u1fc6\u03bd\u03b9\u03bd", "english": "the anger of", "confidence": "sure"},
            {"greek": "\u1f04\u03b5\u03b9\u03b4\u03b5", "english": "Sing", "confidence": "sure"},
            {"greek": "\u03b8\u03b5\u1f70", "english": "O goddess", "confidence": "sure"},
            {"greek": "\u03a0\u03b7\u03bb\u03b7\u03ca\u03ac\u03b4\u03b5\u03c9", "english": "son of Peleus", "confidence": "sure"},
            {"greek": "\u1f08\u03c7\u03b9\u03bb\u1fc6\u03bf\u03c2", "english": "Achilles", "confidence": "sure"}
        ]
    },
    {
        "greek": "\u03bf\u1f50\u03bb\u03bf\u03bc\u03ad\u03bd\u03b7\u03bd \u1f23 \u03bc\u03c5\u03c1\u03af\u1fbd \u1f08\u03c7\u03b1\u03b9\u03bf\u1fd6\u03c2 \u1f04\u03bb\u03b3\u03b5\u1fbd \u1f14\u03b8\u03b7\u03ba\u03b5",
        "english": "that brought countless ills upon the Achaeans",
        "alignment": [
            {"greek": "\u03bf\u1f50\u03bb\u03bf\u03bc\u03ad\u03bd\u03b7\u03bd", "english": "that brought", "confidence": "sure"},
            {"greek": "\u1f23", "english": null, "confidence": "possible"},
            {"greek": "\u03bc\u03c5\u03c1\u03af\u1fbd", "english": "countless", "confidence": "sure"},
            {"greek": "\u1f08\u03c7\u03b1\u03b9\u03bf\u1fd6\u03c2", "english": "upon the Achaeans", "confidence": "sure"},
            {"greek": "\u1f04\u03bb\u03b3\u03b5\u1fbd", "english": "ills", "confidence": "sure"},
            {"greek": "\u1f14\u03b8\u03b7\u03ba\u03b5", "english": "brought", "confidence": "possible"}
        ]
    }
]

def build_alignment_prompt(greek_text, english_text):
    examples = ""
    for ex in FEW_SHOT_EXAMPLES:
        examples += f"""
Example:
Greek:   {ex['greek']}
English: {ex['english']}
Output:
{json_mod.dumps(ex['alignment'], ensure_ascii=False, indent=2)}
"""
    user_msg = f"""{examples}
Now align the following:
Greek:   {greek_text}
English: {english_text}
Output:"""
    return user_msg

print(f"Model: {ANTHROPIC_MODEL}")
print(f"Precision mode: {'sure-only' if ONLY_SURE else 'sure+possible'}")
print(f"Cache file: {ALIGNMENT_CACHE_FILE}")
print("Ready.")


## 13b. Generate alignments (all sections)

This calls the Anthropic API for each section that has both Greek and English.
Results are cached to `poetics_alignments.json` so you only pay for the API once.


In [ ]:
# Load cache if it exists
if os.path.exists(ALIGNMENT_CACHE_FILE):
    with open(ALIGNMENT_CACHE_FILE) as f:
        alignment_cache = json_mod.load(f)
    print(f"Loaded {len(alignment_cache)} cached alignments.")
else:
    alignment_cache = {}

# Collect all (chapter, section) pairs that need alignment
pairs_to_align = []
for ch in sorted(grc_chs | eng_chs, key=section_sort_key):
    g_dict = butcher.get(ch, {})
    e_dict = english.get(ch, {})
    for sec in sorted(set(g_dict.keys()) | set(e_dict.keys()), key=section_sort_key):
        g_text = g_dict.get(sec, '').strip()
        e_text = e_dict.get(sec, '').strip()
        if g_text and e_text:
            key = f"{ch}.{sec}"
            if key not in alignment_cache:
                pairs_to_align.append((ch, sec, g_text, e_text))

print(f"Sections to align: {len(pairs_to_align)}  (already cached: {len(alignment_cache)})")

# Call the API
errors = []
for i, (ch, sec, g_text, e_text) in enumerate(pairs_to_align):
    key = f"{ch}.{sec}"
    user_msg = build_alignment_prompt(g_text, e_text)
    try:
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = resp.content[0].text.strip()
        # Strip markdown fences if present
        if raw.startswith("```"):
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
        alignments = json_mod.loads(raw)
        alignment_cache[key] = alignments
        print(f"  [{i+1}/{len(pairs_to_align)}] {key}: {len(alignments)} pairs")
    except Exception as e:
        errors.append((key, str(e)))
        print(f"  [{i+1}/{len(pairs_to_align)}] {key}: ERROR \u2014 {e}")
    time.sleep(RATE_LIMIT_DELAY)

# Save cache
with open(ALIGNMENT_CACHE_FILE, 'w') as f:
    json_mod.dump(alignment_cache, f, ensure_ascii=False, indent=1)
print(f"\nDone. {len(alignment_cache)} sections aligned, {len(errors)} errors.")
if errors:
    for k, e in errors:
        print(f"  \u26a0 {k}: {e}")


## 13c. Side-by-side display with word-level alignment highlights

Greek and English displayed side by side. **Hover** over a Greek word to highlight its
English counterpart(s) and vice versa. Only **sure** alignments are shown by default
(toggle `ONLY_SURE` above to include possible alignments too).


In [ ]:
import html as html_lib

def make_alignment_id(ch, sec, idx):
    return f"a{ch}_{sec}_{idx}"

def render_aligned_section(ch, sec, g_text, e_text, alignments):
    if not alignments:
        return (f'<td style="vertical-align:top">{html_lib.escape(g_text)}</td>'
                f'<td style="vertical-align:top">{html_lib.escape(e_text)}</td>')

    # Filter by confidence
    filtered = [a for a in alignments if a.get("confidence") == "sure"] if ONLY_SURE else alignments

    g_html = html_lib.escape(g_text)
    e_html = html_lib.escape(e_text)

    pairs = []
    for idx, a in enumerate(filtered):
        gw = a.get("greek", "")
        ew = a.get("english")
        conf = a.get("confidence", "possible")
        if not gw:
            continue
        aid = make_alignment_id(ch, sec, idx)
        pairs.append({"id": aid, "greek": gw, "english": ew, "confidence": conf})

    # Sort by length descending to avoid partial-match clobbering
    for p in sorted(pairs, key=lambda x: -len(x["greek"])):
        gw_esc = html_lib.escape(p["greek"])
        cls = "align-sure" if p["confidence"] == "sure" else "align-possible"
        g_span = (f'<span class="grc-align {cls}" data-pair="{p["id"]}" '
                  f'onmouseenter="hlOn(this)" onmouseleave="hlOff(this)"'
                  f'>{gw_esc}</span>')
        g_html = g_html.replace(gw_esc, g_span, 1)

        if p["english"]:
            ew_esc = html_lib.escape(p["english"])
            e_span = (f'<span class="eng-align {cls}" data-pair="{p["id"]}" '
                      f'onmouseenter="hlOn(this)" onmouseleave="hlOff(this)"'
                      f'>{ew_esc}</span>')
            e_html = e_html.replace(ew_esc, e_span, 1)

    return (f'<td style="vertical-align:top">{g_html}</td>'
            f'<td style="vertical-align:top">{e_html}</td>')


# ── Build the full HTML ──
style_and_script = '''
<style>
.grc-align, .eng-align { cursor: pointer; border-radius: 2px; padding: 0 1px; }
.align-sure { border-bottom: 2px solid #4a90d9; }
.align-possible { border-bottom: 1px dashed #aaa; }
.grc-align:hover, .eng-align:hover,
.grc-align.hl, .eng-align.hl {
    background: #fff3cd; border-bottom-color: #e6a817;
}
</style>
<script>
function hlOn(el) {
    var pair = el.getAttribute('data-pair');
    document.querySelectorAll('[data-pair="'+pair+'"]').forEach(function(e){
        e.classList.add('hl');
    });
}
function hlOff(el) {
    var pair = el.getAttribute('data-pair');
    document.querySelectorAll('[data-pair="'+pair+'"]').forEach(function(e){
        e.classList.remove('hl');
    });
}
</script>
'''

html_parts = [style_and_script]

for ch in sorted(grc_chs | eng_chs, key=section_sort_key):
    g_dict = butcher.get(ch, {})
    e_dict = english.get(ch, {})
    all_secs = sorted(set(g_dict.keys()) | set(e_dict.keys()), key=section_sort_key)

    html_parts.append(
        f'<h3 style="margin-top:24px; border-bottom:2px solid #333; padding-bottom:4px">'
        f'Chapter {ch}</h3>'
        '<table border="1" cellpadding="8" style="border-collapse:collapse; width:100%; table-layout:fixed">'
        '<tr style="background:#f0f0f0"><th style="width:8%">\u00a7</th>'
        '<th style="width:46%">Greek (Butcher)</th>'
        '<th style="width:46%">English (Butcher)</th></tr>')

    for sec in all_secs:
        g_text = g_dict.get(sec, '')
        e_text = e_dict.get(sec, '')
        key = f"{ch}.{sec}"

        if g_text and e_text and key in alignment_cache:
            td_pair = render_aligned_section(ch, sec, g_text, e_text, alignment_cache[key])
        elif g_text and not e_text:
            td_pair = (f'<td style="vertical-align:top; background:#fff5f5">{html_lib.escape(g_text)}</td>'
                       f'<td style="vertical-align:top; background:#fff5f5; color:#aaa; font-style:italic">(no English)</td>')
        elif e_text and not g_text:
            td_pair = (f'<td style="vertical-align:top; background:#f5fff5; color:#aaa; font-style:italic">(no Greek)</td>'
                       f'<td style="vertical-align:top; background:#f5fff5">{html_lib.escape(e_text)}</td>')
        else:
            td_pair = (f'<td style="vertical-align:top">{html_lib.escape(g_text)}</td>'
                       f'<td style="vertical-align:top">{html_lib.escape(e_text)}</td>')

        html_parts.append(
            f'<tr><td style="vertical-align:top; text-align:center; font-weight:bold">{sec}</td>'
            f'{td_pair}</tr>')

    html_parts.append('</table>')

display(HTML('\n'.join(html_parts)))


## 13d. Alignment statistics


In [ ]:
total_pairs = 0
sure_pairs = 0
possible_pairs = 0
null_english = 0
sections_aligned = len(alignment_cache)

for key, als in alignment_cache.items():
    for a in als:
        total_pairs += 1
        if a.get("confidence") == "sure":
            sure_pairs += 1
        else:
            possible_pairs += 1
        if not a.get("english"):
            null_english += 1

print(f"Sections aligned:      {sections_aligned}")
print(f"Total alignment pairs: {total_pairs}")
print(f"  Sure:                {sure_pairs} ({100*sure_pairs/max(total_pairs,1):.1f}%)")
print(f"  Possible:            {possible_pairs} ({100*possible_pairs/max(total_pairs,1):.1f}%)")
print(f"  Null (no English):   {null_english} ({100*null_english/max(total_pairs,1):.1f}%)")
print(f"\nPrecision mode: {'sure-only' if ONLY_SURE else 'sure+possible'}")


## 14. Translation coverage statistics

In [86]:
total_grc = sum(len(ch) for ch in butcher.values())
total_eng = sum(len(ch) for ch in english.values())
aligned, only_grc, only_eng = 0, 0, 0

for ch in sorted(grc_chs | eng_chs, key=section_sort_key):
    g_secs = set(butcher.get(ch, {}).keys())
    e_secs = set(english.get(ch, {}).keys())
    aligned += len(g_secs & e_secs)
    only_grc += len(g_secs - e_secs)
    only_eng += len(e_secs - g_secs)

print('=' * 60)
print('PART B — GREEK ↔ ENGLISH SUMMARY')
print('=' * 60)
print(f'\n  Greek sections:     {total_grc}')
print(f'  English sections:   {total_eng}')
print(f'  Aligned pairs:      {aligned}')
print(f'  Only in Greek:      {only_grc}')
print(f'  Only in English:    {only_eng}')
if only_grc == 0 and only_eng == 0:
    print('\n✅ Perfect section alignment')
else:
    if only_grc > 0:
        print(f'\n⚠️  {only_grc} Greek sections have no English translation')
    if only_eng > 0:
        print(f'⚠️  {only_eng} English sections have no Greek counterpart')

PART B — GREEK ↔ ENGLISH SUMMARY

  Greek sections:     381
  English sections:   381
  Aligned pairs:      381
  Only in Greek:      0
  Only in English:    0

✅ Perfect section alignment
